# AutoGen 工具使用示例

## 导入所需包

In [4]:
import os
import json

import requests
from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken
from autogen_core.tools import FunctionTool
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console
from typing import Any, Callable, Set, Dict, List, Optional

from dotenv import load_dotenv  # 用于安全加载环境变量
load_dotenv()

True

## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`model` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场上可用的其他模型，以查看不同的结果。

作为快速测试，我们将运行一个简单的提示 - `法国的首都是什么`。

In [5]:
client = AzureAIChatCompletionClient(
    model=os.getenv("MODEL_FREE_8B"),
    endpoint=os.getenv("API_URL"),
    # 要向模型进行身份验证，您需要在 GitHub 设置中生成个人访问令牌 (PAT)。
    # 按照此处的说明创建 PAT 令牌：https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

result = await client.create([UserMessage(content="What is the capital of France?", source="user")])
print(result)

/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/autogen_ext/models/azure/_azure_ai_client.py:307: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(config["model_info"])


finish_reason='stop' content='The capital of France is Paris.' usage=RequestUsage(prompt_tokens=19, completion_tokens=7) cached=False logprobs=None thought=None


## 定义函数

在本示例中，我们将为代理提供一个工具，该工具是一个函数，包含可用的度假目的地列表及其可用性。

您可以认为这是一个旅行代理可能有权访问旅行数据库的场景。

在浏览此示例时，请随意尝试定义代理可以调用的新函数和工具。

In [6]:
from typing import Dict, List, Optional


def vacation_destinations(city: str) -> tuple[str, str]:
    """
    检查特定度假目的地是否可用
    
    参数:
        city (str): 要检查的城市名称
        
    返回:
        tuple: 包含城市名称和可用性状态（'Available' 或 'Unavailable'）
    """
    destinations = {
        "Barcelona": "Available",
        "Tokyo": "Unavailable",
        "Cape Town": "Available",
        "Vancouver": "Available",
        "Dubai": "Unavailable",
    }

    if city in destinations:
        return city, destinations[city]
    else:
        return city, "City not found"

# 示例用法:
# city, status = vacation_destinations("Barcelona")
# print('How about visiting {}? It's currently {} there!'.format(city, status))

## 定义函数工具
要让代理将 `vacation_destinations` 用作 `FunctionTool`，我们需要将其定义为一个。

我们还将提供工具的描述，这有助于代理确定该工具在用户请求的任务中的用途。

In [7]:
get_vacations = FunctionTool(
    vacation_destinations, description="Search for vacation destinations and if they are available or not."
)

## 定义代理

现在我们可以在下面的代码中创建代理。我们定义 `system_message` 来给代理提供有关如何帮助用户找到度假目的地的指示。

我们还将 `reflect_on_tool_use` 参数设置为 true。这允许使用 LLM 获取工具调用的响应并使用自然语言发送响应。

您可以将该参数设置为 false 以查看差异。

In [8]:
agent = AssistantAgent(
    name="assistant",
    model_client=client,
    tools=[get_vacations],
    system_message="You are a travel agent that helps users find vacation destinations.",
    reflect_on_tool_use=True,
)

## 运行代理

现在我们可以使用要求去东京旅行的初始用户消息运行代理。

您可以更改此城市目的地，以查看代理如何响应该城市的可用性。

In [9]:
async def assistant_run() -> None:
    response = await agent.on_messages(
        [TextMessage(content="I would like to take a trip to Tokyo", source="user")],
        cancellation_token=CancellationToken(),
    )
    print(response.inner_messages)
    print(response.chat_message)


# Use asyncio.run(assistant_run()) when running in a script.
await assistant_run()

[ToolCallRequestEvent(id='db412865-81c8-4ce1-babb-fa2970051f75', source='assistant', models_usage=RequestUsage(prompt_tokens=194, completion_tokens=22), metadata={}, created_at=datetime.datetime(2026, 2, 4, 7, 43, 3, 528152, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='019c279ad6d3ddba3e0309c03916431c', arguments='{"city": "Tokyo"}', name='vacation_destinations')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='4ac1d141-b6d1-4f6a-940f-27d4de200ccb', source='assistant', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 2, 4, 7, 43, 3, 530104, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content="('Tokyo', 'Unavailable')", name='vacation_destinations', call_id='019c279ad6d3ddba3e0309c03916431c', is_error=False)], type='ToolCallExecutionEvent')]
id='94a9735a-38f1-49b9-9cac-9d08d7e0f32a' source='assistant' models_usage=RequestUsage(prompt_tokens=80, completion_tokens=64) metadata={} created_at=datetime.datetime(2026, 2, 4, 7, 43, 8,